# EDA 01 — SID español para clasificación de tickets

**Objetivo académico:** explorar el dataset tras aplicar el mapeo preliminar de 44 intenciones a seis departamentos, **antes de entrenar ML**.

Requisitos: `python -m pip install pandas matplotlib jupyter`. Generar antes los CSV con `python src/data/prepare_sid.py`.

Este notebook examina distribución, longitudes, palabras frecuentes, faltantes, duplicados exactos y ejemplos por clase. **No implementa aún el pipeline NLP ni entrena modelos.**


In [ ]:
from pathlib import Path
from collections import Counter
import re
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed" / "sid"
splits = ["train", "validation", "validation_real", "test"]
frames = []
for split in splits:
    file = DATA / f"{split}_es.csv"
    if not file.exists():
        raise FileNotFoundError(f"Falta {file}. Ejecuta python src/data/prepare_sid.py desde la raíz.")
    frame = pd.read_csv(file)
    frame["original_split"] = split
    frames.append(frame)
df = pd.concat(frames, ignore_index=True)
print("Registros totales aceptados:", len(df))
display(df.head(5))


## 1. Distribución por split y departamento

Analizaremos el **desbalance**, especialmente Marketing. No se debe equilibrar el conjunto de prueba artificialmente.

In [ ]:
by_split = pd.crosstab(df["original_split"], df["department"])
display(by_split)
by_department = df["department"].value_counts().rename_axis("departamento").reset_index(name="cantidad")
by_department["porcentaje"] = (by_department["cantidad"] / len(df) * 100).round(2)
display(by_department)
ax = df["department"].value_counts().plot.bar(figsize=(10, 4), title="Mensajes por departamento")
ax.set_xlabel("Departamento")
ax.set_ylabel("Mensajes")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()


## 2. Valores faltantes y longitud

El PDF requiere analizar valores ausentes y extensión de los textos. Los mensajes de SID son predominantemente cortos; evaluar después ejemplos propios más extensos.

In [ ]:
display(df.isna().sum().to_frame("faltantes"))
df["word_count"] = df["text"].fillna("").astype(str).str.split().str.len()
df["char_count"] = df["text"].fillna("").astype(str).str.len()
display(df.groupby("department")["word_count"].agg(["count", "mean", "min", "median", "max"]).round(2))
ax = df["word_count"].plot.hist(bins=20, figsize=(9, 4), title="Longitud de mensajes (palabras)")
ax.set_xlabel("Cantidad de palabras")
plt.tight_layout()
plt.show()


## 3. Palabras frecuentes (exploración, todavía sin stop words)

Este conteo exploratorio **no sustituye** la tokenización ni el filtrado de stop words del pipeline de NLP. Se usará luego para explicar decisiones de preprocesamiento.

In [ ]:
def palabras_crudas(text):
    return re.findall(r"\b\w+\b", str(text).casefold(), flags=re.UNICODE)

frecuencias = Counter(token for text in df["text"].dropna() for token in palabras_crudas(text))
display(pd.DataFrame(frecuencias.most_common(25), columns=["palabra", "frecuencia"]))


## 4. Duplicados exactos y riesgo de frases parecidas

Se revisan duplicados que permanezcan tras la limpieza preliminar. **Cero duplicados exactos no descarta paráfrasis casi idénticas** entre entrenamiento y prueba.

In [ ]:
df["text_key"] = (df["text"].fillna("").astype(str).str.casefold()
                   .str.replace(r"\s+", " ", regex=True).str.strip())
print("Duplicados exactos restantes:", df.duplicated("text_key").sum())
print("Ejemplos por split:")
display(df.groupby("original_split").size().to_frame("cantidad"))
print("Número de clases:", df["department"].nunique())


## 5. Revisión humana de muestras

Leer al menos tres ejemplos al azar por categoría para identificar intenciones mal asignadas, frases repetitivas y textos artificiales. Aumentar la muestra en una revisión formal.

In [ ]:
for departamento, grupo in df.groupby("department"):
    print(f"\n### {departamento}")
    display(grupo[["text", "intent", "original_split"]]
            .sample(n=min(3, len(grupo)), random_state=42))


## 6. Resumen original del proceso de preparación

In [ ]:
resumen = DATA / "summary.json"
if resumen.exists():
    info = json.loads(resumen.read_text(encoding="utf-8"))
    print("Estado del mapeo:", info.get("mapping_status"))
    print("Intenciones mapeadas:", info.get("mapped_intents"))
    print("Registros de Marketing:", info.get("all_splits_by_department", {}).get("Marketing"))
    display(pd.DataFrame(info.get("splits", []))[["split", "counts", "by_department"]])
else:
    print("No se encontró summary.json")


## 7. Datos propios: revisión separada

La plantilla `data/templates/custom_messages.csv` es solo un ejemplo de esquema. El dataset propio (si existe) se mantiene **aparte** hasta establecer control de calidad y evitar fugas entre entrenamiento y evaluación.

In [ ]:
propios = ROOT / "data" / "raw" / "custom" / "messages.csv"
if propios.exists():
    own = pd.read_csv(propios)
    print("Registros propios:", len(own))
    display(own["department"].value_counts(dropna=False))
    display(own.head(5))
else:
    print(f"Sin dataset propio todavía ({propios}); esta etapa queda pendiente.")


## Conclusiones que el equipo debe redactar

1. ¿Qué clase está más desbalanceada?
2. ¿La longitud de los textos basta para identificar un departamento?
3. ¿Qué intenciones de SID se asignaron incorrectamente?
4. ¿Cuántos ejemplos propios son viables sin duplicar plantillas?
5. ¿Qué medidas adoptaremos para evitar contaminación entre splits?

**Siguiente fase académica (después de aprobar el dataset):** normalización, tokenización, stop words, stemming vs. lematización, n-gramas y TF-IDF con justificación y comparación.